In [21]:
import sys, subprocess, pkgutil, os

def ensure_pyg():
    import torch
    if pkgutil.find_loader("torch_geometric") is not None:
        print("torch_geometric already available ✅")
        return
    ver = torch.__version__.split('+')[0]   # e.g. '2.9.0'
    cu  = torch.version.cuda               # e.g. '12.8'
    if cu is None:
        wheel_idx = f"torch-{ver}+cpu.html"
    else:
        cu_tag = "cu" + cu.replace('.', '')   # 'cu128'
        wheel_idx = f"torch-{ver}+{cu_tag}.html"
    idx_url = f"https://data.pyg.org/whl/{wheel_idx}"
    print("[INFO] Installing PyG wheels for", ver, cu, "from", idx_url)
    cmds = [
        [sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"],
        [sys.executable, "-m", "pip", "install", "pyg_lib", "torch_scatter", "torch_sparse", "torch_cluster", "torch_spline_conv", "-f", idx_url],
        [sys.executable, "-m", "pip", "install", "torch_geometric"],
    ]
    for c in cmds:
        print("->", " ".join(c))
        subprocess.check_call(c)

try:
    ensure_pyg()
    import torch_geometric
    print("✅ torch_geometric version:", torch_geometric.__version__)
except Exception as e:
    print("❌ Failed to prepare torch_geometric:", e)
    print("  - カーネルが別環境の可能性があります。上のセル出力のURL/コマンドでターミナル側に先に入れてから、")
    print("    Notebookのカーネルをそのvenvに切り替えてください（Kernel → Change Kernel）。")
    raise



torch_geometric already available ✅
✅ torch_geometric version: 2.7.0


In [22]:
from pathlib import Path

in_dir = Path("/home/yudai/Project/research/Vul_Detection/data/input")
files = sorted(in_dir.glob("*_cpg_input.pkl"))
print("Found:", len(files))
for i, f in enumerate(files, 1):
    print(f"{i}. {f.name}")
assert files, "data/input/*_cpg_input.pkl が見つかりません。"


Found: 2
1. 0_cpg_input.pkl
2. 1_cpg_input.pkl


In [23]:
# --- shim: map numpy._core.* -> numpy.core.* for unpickling ---
import sys, types, importlib
import numpy as _np

_aliases = {
    "numpy._core": "numpy.core",
    "numpy._core.numeric": "numpy.core.numeric",
    "numpy._core.fromnumeric": "numpy.core.fromnumeric",
    "numpy._core.multiarray": "numpy.core.multiarray",
    "numpy._core._multiarray_umath": "numpy.core._multiarray_umath",
    "numpy._core.records": "numpy.core.records",
}

for new, old in _aliases.items():
    try:
        mod = importlib.import_module(old)
    except Exception:
        # 一部は存在しない場合があるが、numeric と fromnumeric があれば十分なことが多い
        continue
    parent = new.rsplit(".", 1)[0]
    if parent not in sys.modules:
        sys.modules[parent] = types.ModuleType(parent)
    sys.modules[new] = mod

print("✅ NumPy shim active. Now you can pd.read_pickle(...) safely.")


✅ NumPy shim active. Now you can pd.read_pickle(...) safely.


In [ ]:
import pandas as pd
import torch
from torch_geometric.data import Data
from collections import Counter, defaultdict

# ここは既存の files リストを利用
pkl0 = files[0]
pkl1 = files[1]

df0 = pd.read_pickle(pkl0)
df1 = pd.read_pickle(pkl1)

print(f"[df0] Columns: {list(df0.columns)}  shape={df0.shape}")
print(f"[df1] Columns: {list(df1.columns)}  shape={df1.shape}")

display(df0.head(1))
display(df1.head(1))

# -------- 安全チェック --------
for name, df in (("df0", df0), ("df1", df1)):
    assert {"input","target","func"}.issubset(df.columns), f"{name}: 必要列（input/target/func）がありません。"
    assert isinstance(df.iloc[0]["input"], Data), f"{name}: input が torch_geometric.data.Data ではありません。"

# ========== ユーティリティ ==========
def _shape_of(x):
    try:
        if hasattr(x, "shape"):
            return tuple(x.shape)
        if hasattr(x, "size"):
            return x.size()
        return len(x)
    except Exception:
        return None

def _dtype_of(x):
    try:
        return getattr(x, "dtype", type(x))
    except Exception:
        return type(x)

def _to_list_safe(t, k=5):
    try:
        if torch.is_tensor(t):
            return t[:k].tolist()
        if isinstance(t, (list, tuple)):
            return list(t)[:k]
    except Exception:
        pass
    return None

def list_data_attributes(d: Data):
    print("---- attributes ----")
    # keys をメソッド呼び出しで取得
    try:
        keys_list = list(d.keys())  # ← ここが重要
    except TypeError:
        # 万一 keys が property / 互換性のためのフォールバック
        keys_list = list(getattr(d, "keys", [])) if not callable(getattr(d, "keys", None)) else []
    except Exception:
        keys_list = list(getattr(d, "__dict__", {}).keys())

    print("keys:", keys_list)
    for k in keys_list:
        v = getattr(d, k, None)
        # shape / dtype の表示は既存ヘルパを利用
        def _shape_of(x):
            try:
                if hasattr(x, "shape"):
                    return tuple(x.shape)
                if hasattr(x, "size"):
                    return x.size()
                return len(x)
            except Exception:
                return None

        def _dtype_of(x):
            try:
                return getattr(x, "dtype", type(x))
            except Exception:
                return type(x)

        print(f" - {k:12s} type={type(v)} shape={_shape_of(v)} dtype={_dtype_of(v)}")


def check_graph_invariants(d: Data):
    print("\n---- invariants ----")
    # num_nodes 推定
    n_nodes = getattr(d, "num_nodes", None)
    if n_nodes is None and hasattr(d, "x") and d.x is not None:
        n_nodes = int(d.x.size(0))
        print(f"num_nodes inferred from x: {n_nodes}")
    else:
        print(f"num_nodes(attr): {n_nodes}")

    # edge_index
    if not (hasattr(d, "edge_index") and d.edge_index is not None):
        print("edge_index: None")
        return
    ei = d.edge_index
    if not torch.is_tensor(ei):
        print("edge_index is not torch.Tensor")
        return
    if ei.dim() != 2 or ei.size(0) != 2:
        print("edge_index shape unexpected:", tuple(ei.shape))
        return

    m = int(ei.size(1))
    print("num_edges:", m)

    # 範囲外・自己ループ・重複・孤立ノード
    src = ei[0]
    dst = ei[1]

    out_of_range = None
    if n_nodes is not None:
        out_of_range = ((src < 0) | (src >= n_nodes) | (dst < 0) | (dst >= n_nodes)).nonzero().numel()
        print("out_of_range edges:", int(out_of_range))

    self_loops = (src == dst).sum().item()
    print("self_loops:", int(self_loops))

    # 重複（厳密）
    try:
        pairs = torch.stack([src, dst], dim=1)
        # unique_rows で重複数を出す
        uniq_pairs, counts = torch.unique(pairs, dim=0, return_counts=True)
        dup_edges = int((counts > 1).sum().item())
        print("duplicate edge pairs (# unique that appear >1):", dup_edges)
    except Exception as e:
        print("duplicate check skipped:", repr(e))

    # 孤立ノード
    if n_nodes is not None:
        deg = torch.bincount(src, minlength=n_nodes) + torch.bincount(dst, minlength=n_nodes)
        isolated = int((deg == 0).sum().item())
        print("isolated_nodes:", isolated)

    # edge_type 整合性と分布
    if hasattr(d, "edge_type") and d.edge_type is not None and torch.is_tensor(d.edge_type):
        et = d.edge_type
        if et.numel() == m:
            uniq, cnt = torch.unique(et, return_counts=True)
            print("edge_type counts:", dict(zip([int(u) for u in uniq.tolist()],
                                                [int(c) for c in cnt.tolist()])))
        else:
            print(f"edge_type length mismatch: edge_type={et.numel()} vs edges={m}")
    else:
        print("edge_type: None")

def preview_x_and_stats(d: Data, *, head=5, stat_cols=16):
    if not (hasattr(d, "x") and d.x is not None):
        print("\n---- x preview ----\n(x is None)")
        return
    X = d.x
    print("\n---- x preview ----")
    print("x.shape:", tuple(X.shape), "dtype:", X.dtype)
    print("x[:5]:")
    try:
        print(X[:head])
    except Exception as e:
        print("print x head failed:", repr(e))

    # 簡易統計（先頭 stat_cols 列のみ）
    try:
        cols = min(stat_cols, X.size(1)) if X.dim() == 2 else 0
        if cols > 0:
            x_slice = X[:, :cols].float()
            nan_ct = torch.isnan(x_slice).sum().item()
            print(f"NaN in first {cols} cols:", nan_ct)
            col_min = torch.nan_to_num(x_slice, nan=0.0).min(dim=0).values
            col_max = torch.nan_to_num(x_slice, nan=0.0).max(dim=0).values
            print("col_min[:8]:", col_min[:8].tolist())
            print("col_max[:8]:", col_max[:8].tolist())
    except Exception as e:
        print("x stats failed:", repr(e))

def preview_source_mapping(d: Data, func_src: str, *, pick_index=0):
    print("\n---- code mapping sample ----")
    if not isinstance(func_src, str) or not func_src.strip():
        print("(func src not available)")
        return
    code_lines = func_src.splitlines()
    # 優先して line を使う
    if hasattr(d, "line") and d.line is not None and torch.is_tensor(d.line) and d.line.numel() > pick_index:
        li = int(d.line[pick_index].item())
        if 1 <= li <= len(code_lines):
            print(f"line[{pick_index}] -> code line {li}:")
            print(code_lines[li-1])
            return
    print("(no line mapping found)")

def describe_data_obj(d: Data, *, name="data", func_src=None):
    print(f"\n===== {name}: torch_geometric.data.Data =====")
    print(d)  # Data(...) summary

    # ここは d.keys() を呼び出す
    try:
        print("keys (summary):", list(d.keys()))
    except Exception:
        print("keys (summary): <unavailable>")

    list_data_attributes(d)
    check_graph_invariants(d)
    preview_x_and_stats(d)
    if func_src is not None:
        preview_source_mapping(d, func_src)


def preview_func(src: str, *, lines=12):
    print("\n===== func preview (first", lines, "lines) =====")
    if not isinstance(src, str):
        print("(func is not string)")
        return
    snippet = src.splitlines()[:lines]
    for i, ln in enumerate(snippet, 1):
        print(f"{i:3d}: {ln}")

def inspect_df(df: pd.DataFrame, *, name="df", row_idx=0):
    print(f"\n\n################ {name} ################")
    print("shape:", df.shape)
    print("target value counts:\n", df["target"].value_counts())
    print("func length stats:\n", df["func"].str.len().describe())

    row = df.iloc[row_idx]
    d: Data = row["input"]
    print(f"\n-- sample row index={row_idx} target={row['target']}")
    preview_func(row["func"], lines=12)
    describe_data_obj(d, name=f"{name}[{row_idx}].input", func_src=row["func"])

def scan_dataframe(df: pd.DataFrame, *, name="df"):
    print(f"\n\n================ scan: {name} ================")
    n = len(df)
    print("rows:", n)
    tgt = df["target"].value_counts()
    print("target counts:", dict(tgt))

    # 集計
    nodes_stats = []
    edges_stats = []
    xdim_stats = Counter()
    etype_counter = Counter()
    missing_edge_type = 0
    edge_type_mismatch = 0
    self_loops_total = 0
    isolated_total = 0
    out_of_range_total = 0
    dup_edges_total = 0

    for i, d in enumerate(df["input"]):
        if not isinstance(d, Data):
            continue

        # num_nodes
        nn = getattr(d, "num_nodes", None)
        if nn is None and hasattr(d, "x") and d.x is not None:
            nn = int(d.x.size(0))
        nodes_stats.append(nn if nn is not None else -1)

        # edges
        m = None
        if hasattr(d, "edge_index") and d.edge_index is not None and torch.is_tensor(d.edge_index) and d.edge_index.dim()==2:
            m = int(d.edge_index.size(1))
            edges_stats.append(m)

            src = d.edge_index[0]
            dst = d.edge_index[1]

            # self loops
            self_loops_total += int((src == dst).sum().item())

            # out of range
            if nn is not None:
                out_of_range_total += int(((src < 0) | (src >= nn) | (dst < 0) | (dst >= nn)).sum().item())

            # dup edges
            try:
                pairs = torch.stack([src, dst], dim=1)
                _, counts = torch.unique(pairs, dim=0, return_counts=True)
                dup_edges_total += int((counts > 1).sum().item())
            except Exception:
                pass

            # isolated nodes
            if nn is not None:
                deg = torch.bincount(src, minlength=nn) + torch.bincount(dst, minlength=nn)
                isolated_total += int((deg == 0).sum().item())

        # x dim
        if hasattr(d, "x") and d.x is not None and d.x.dim() == 2:
            xdim_stats.update([int(d.x.size(1))])

        # edge_type
        if hasattr(d, "edge_type") and d.edge_type is not None and torch.is_tensor(d.edge_type):
            if m is not None and d.edge_type.numel() != m:
                edge_type_mismatch += 1
            else:
                uniq, cnt = torch.unique(d.edge_type, return_counts=True)
                etype_counter.update(dict(zip([int(u) for u in uniq.tolist()],
                                              [int(c) for c in cnt.tolist()])))
        else:
            missing_edge_type += 1

    import math
    def _summary(arr):
        arr = [x for x in arr if x is not None and x >= 0]
        if not arr:
            return {"count": 0}
        return {
            "count": len(arr),
            "min": int(min(arr)),
            "max": int(max(arr)),
            "mean": float(sum(arr)/len(arr))
        }

    print("nodes summary:", _summary(nodes_stats))
    print("edges summary:", _summary(edges_stats))
    print("x feature dims (frequency):", dict(xdim_stats))
    print("edge_type total counts:", dict(etype_counter))
    print("edge_type missing rows:", int(missing_edge_type))
    print("edge_type length mismatch rows:", int(edge_type_mismatch))
    print("self_loops total:", int(self_loops_total))
    print("out_of_range edges total:", int(out_of_range_total))
    print("isolated_nodes total:", int(isolated_total))
    print("duplicate edge pairs total:", int(dup_edges_total))

def compare_two_dfs(dfA: pd.DataFrame, dfB: pd.DataFrame, nameA="df0", nameB="df1"):
    print(f"\n\n================ compare: {nameA} vs {nameB} ================")
    print(f"{nameA} rows={len(dfA)}  {nameB} rows={len(dfB)}")
    print(f"{nameA} target ratio:\n{dfA['target'].value_counts(normalize=True)}")
    print(f"{nameB} target ratio:\n{dfB['target'].value_counts(normalize=True)}")

    # ノード・エッジ平均（簡易）
    def _avg_nodes_edges(df):
        nodes, edges = [], []
        for d in df["input"]:
            if not isinstance(d, Data): continue
            nn = getattr(d, "num_nodes", None)
            if nn is None and hasattr(d, "x") and d.x is not None:
                nn = int(d.x.size(0))
            if nn is not None: nodes.append(nn)
            if hasattr(d, "edge_index") and d.edge_index is not None and torch.is_tensor(d.edge_index) and d.edge_index.dim()==2:
                edges.append(int(d.edge_index.size(1)))
        def _avg(a): return (sum(a)/len(a)) if a else float("nan")
        return _avg(nodes), _avg(edges)

    A_nodes_mean, A_edges_mean = _avg_nodes_edges(dfA)
    B_nodes_mean, B_edges_mean = _avg_nodes_edges(dfB)
    print(f"{nameA} mean nodes={A_nodes_mean:.2f}  mean edges={A_edges_mean:.2f}")
    print(f"{nameB} mean nodes={B_nodes_mean:.2f}  mean edges={B_edges_mean:.2f}")

# -------- 実行（必要なら row_idx を変えて確認） --------
inspect_df(df0, name="df0", row_idx=0)
inspect_df(df1, name="df1", row_idx=0)

# データフレーム全体の集計
scan_dataframe(df0, name="df0")
scan_dataframe(df1, name="df1")

# 2つの比較
compare_two_dfs(df0, df1, nameA="df0", nameB="df1")

# 可視化フック（任意）：utils.validate.code_map.render_code_and_graph_html がある場合だけ動かす
try:
    from utils.validate.code_map import render_code_and_graph_html
    d0 = df0.iloc[0]["input"]
    html_path = render_code_and_graph_html(
        d0,
        out_html="data/inspect/graph_with_code_df0.html",
        edge_type_labels={0:"AST",1:"CFG",2:"PDG"},
        title="df0 sample"
    )
    print("[rendered]", html_path)
except Exception as e:
    print("[viz skipped]", repr(e))


[df0] Columns: ['input', 'target', 'func']  shape=(965, 3)
[df1] Columns: ['input', 'target', 'func']  shape=(990, 3)


,input,target,func
0,"[(x, [tensor([ 0.0000e+00, -1.4232e-01, 3.667...",1,"glue(cirrus_bitblt_rop_fwd_, ROP_NAME)(CirrusV..."


,input,target,func
0,"[(x, [tensor([ 0.0000e+00, -1.4232e-01, 3.667...",0,static uint64_t virtio_net_guest_offloads_by_f...




################ df0 ################
shape: (965, 3)
target value counts:
 target
1    501
0    464
Name: count, dtype: int64
func length stats:
 count     965.000000
mean      511.589637
std       315.007613
min        54.000000
25%       242.000000
50%       441.000000
75%       741.000000
max      1198.000000
Name: func, dtype: float64

-- sample row index=0 target=1

===== func preview (first 12 lines) =====
  1: glue(cirrus_bitblt_rop_fwd_, ROP_NAME)(CirrusVGAState *s,
  2:                              uint8_t *dst,const uint8_t *src,
  3:                              int dstpitch,int srcpitch,
  4:                              int bltwidth,int bltheight)
  5: {
  6:     int x,y;
  7:     dstpitch -= bltwidth;
  8:     srcpitch -= bltwidth;
  9:     for (y = 0; y < bltheight; y++) {
 10:         for (x = 0; x < bltwidth; x++) {
 11:             ROP_OP(*dst, *src);
 12:             dst++;

===== df0[0].input: torch_geometric.data.Data =====
Data(x=[500, 769], edge_index=[2, 12],

TypeError: 'method' object is not iterable